# ImageNet Training Loop

In this notebook, we will run an ImageNet training loop on a single GPU in AWS. Eventually we will create a .py file to train ImageNet on multiple GPUs in AWS or RunPod.

In [ ]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '../..'))

In [ ]:
import torch
import torchvision
from torch.optim import SGD
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR

from utils.imagenet import get_train_transform, get_val_transform
from utils.metrics import accuracy, topk_accuracy

In [ ]:
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

In [ ]:
# Hyperparameters

# Use bucket mount
bucket_path = 'mnt/imagenet'

# single epoch for testing. Eventually will set to 90
num_epochs = 1

# may need to be smaller if OOM occurs
batch_size = 128

# Change depending on number of CPUs, optimize
num_workers = 4
pin_memory = True

# How many batches before logging loss
log_every = 1000

# How many times to perform validation per epoch
val_per_epoch = 2

# Optimizer hyperparameters
opt_kwargs = {'lr': 0.1 * batch_size/256, 'momentum': 0.9}
num_warmup = 5

# Create model

In [ ]:
model = torchvision.models.resnet50().to(device)

# Import dataset

In [ ]:
train_ds = torchvision.datasets.ImageFolder(bucket_path + "/train", transform=get_train_transform())
val_ds = torchvision.datasets.ImageFolder(bucket_path + "/val", transform=get_val_transform())

# Create dataloader

In [ ]:
train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, 
    num_workers=num_workers, pin_memory=pin_memory, drop_last=True
)
print(f"Length of training dataloader is {len(train_dl)}")

In [ ]:
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=2*batch_size, shuffle=False,
    num_workers=num_workers, pin_memory=pin_memory, drop_last=False
)
print(f"Length of validation dataloader is {len(val_dl)}")

# Introduce loss and training metrics

In [ ]:
loss_fn = torch.nn.functional.cross_entropy
metrics = [accuracy, topk_accuracy]
train_metrics = [[]] * (1 + len(metrics))
val_metrics = [[]] * (1 + len(metrics))

# Create optimizer

In [ ]:
opt = SGD(model.parameters(), **opt_kwargs)

# Create learning rate scheduler

In [ ]:
scheduler1 = LinearLR(opt, 0.01, 1.0, num_warmup)
scheduler2 = CosineAnnealingLR(opt, T_max=max(25, num_epochs - num_warmup), eta_min=1e-5)
scheduler = SequentialLR(opt, schedulers=[scheduler1, scheduler2], milestones=[num_warmup])

# Create loss scaler for mixed-precision training

In [ ]:
scaler = torch.amp.grad_scaler(device.type)

# Write training loop

In [ ]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    for i, (X, y) in enumerate(train_dl):
        
        with torch.autocast(device.type):
            logits = model(X)
            loss = loss_fn(logits, y)
            
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)